# 01 — Read Model: Extract Text from Scanned Claims

**Time**: ~5 minutes · **Model**: `prebuilt-read` · **No training required**

---

## Insurance Scenario

A policyholder submits a **scanned claim letter** — possibly handwritten or a mix of printed and handwritten text. Before any processing begins, you need to extract all the text reliably.

The **Read model** is your starting point for any text extraction task:
- Extracts printed and handwritten text
- Detects languages
- Provides word-level confidence scores
- Works with PDFs, images, and Office files

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze a Document with the Read Model

We'll use a sample document. In production, this would be a scanned claim letter, medical report, or handwritten note from a policyholder.

In [ ]:
# Sample document URL (replace with your own scanned claim letter)
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

# Submit for analysis
poller = client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=document_url)
)
result: AnalyzeResult = poller.result()

print(f"Pages analyzed: {len(result.pages)}")
print(f"Total characters extracted: {len(result.content)}")

## View the Full Extracted Text

The `content` property contains all the text in reading order — useful for feeding into downstream systems like a claims management platform.

In [ ]:
# Print the full extracted text (first 1000 chars for brevity)
print("=" * 60)
print("EXTRACTED TEXT (first 1000 characters)")
print("=" * 60)
print(result.content[:1000])
print("...")

## Examine Per-Page Details

Each page contains detailed information about lines, words, and their positions.

In [ ]:
for page in result.pages:
    print(f"\n--- Page {page.page_number} ---")
    print(f"  Dimensions: {page.width} x {page.height} ({page.unit})")
    print(f"  Lines: {len(page.lines) if page.lines else 0}")
    print(f"  Words: {len(page.words) if page.words else 0}")
    print(f"  Selection marks: {len(page.selection_marks) if page.selection_marks else 0}")
    
    # Show first 5 lines
    if page.lines:
        print(f"\n  First 5 lines:")
        for line in page.lines[:5]:
            print(f"    \"{line.content}\"")

## Word-Level Confidence Scores

**Why this matters for insurance**: Confidence scores let you flag uncertain extractions for human review. For example, if a handwritten claim amount has low confidence, route it to an adjuster.

Confidence ranges from 0.0 (uncertain) to 1.0 (highly confident).

In [ ]:
import pandas as pd

# Collect word-level confidence data from page 1
page = result.pages[0]
word_data = []
if page.words:
    for word in page.words[:20]:  # First 20 words
        word_data.append({
            "Word": word.content,
            "Confidence": f"{word.confidence:.2%}",
            "Flag": "⚠️ Review" if word.confidence < 0.80 else "✅ OK"
        })

df = pd.DataFrame(word_data)
print("Word-level confidence scores (first 20 words):")
print(df.to_string(index=False))

## Detect Handwriting

**Why this matters for insurance**: Many claim forms include handwritten sections (e.g., accident descriptions, signatures). The Read model detects handwritten vs. printed text.

In [ ]:
if result.styles:
    has_handwriting = any(style.is_handwritten for style in result.styles)
    if has_handwriting:
        print("✍️ Document contains HANDWRITTEN content")
        for style in result.styles:
            if style.is_handwritten:
                for span in style.spans:
                    handwritten_text = result.content[span.offset:span.offset + span.length]
                    print(f"   Handwritten text: \"{handwritten_text[:100]}...\"")
    else:
        print("📄 Document contains only PRINTED text")
else:
    print("📄 No style information detected (likely all printed text)")

## Analyze a Local File

In production, documents often come as file uploads. Here's how to analyze a local file:

In [ ]:
# Example: Analyze a local file (uncomment and update path)
#
# with open("../sample-documents/your-claim-letter.pdf", "rb") as f:
#     poller = client.begin_analyze_document("prebuilt-read", body=f)
#     result = poller.result()
#     print(f"Extracted {len(result.content)} characters from local file")

print("Uncomment the code above and update the file path to analyze your own documents.")

## Generate a Searchable PDF

The Read model can convert scanned PDFs into **searchable PDFs** with embedded text — useful for archival and compliance.

In [ ]:
from azure.ai.documentintelligence.models import AnalyzeOutputOption

# Analyze with PDF output
poller = client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=document_url),
    output=[AnalyzeOutputOption.PDF]
)
result = poller.result()
operation_id = poller.details["operation_id"]

# Download the searchable PDF
response = client.get_analyze_result_pdf(
    model_id=result.model_id,
    result_id=operation_id
)

output_path = "../sample-documents/searchable_output.pdf"
with open(output_path, "wb") as writer:
    writer.writelines(response)

print(f"✅ Searchable PDF saved to {output_path}")
print("   This PDF has embedded text — you can now search, copy, and index it.")

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Text extraction** | Digitize scanned claim letters, medical reports, correspondence |
| **Handwriting detection** | Identify handwritten notes and descriptions on claim forms |
| **Word confidence** | Flag uncertain text for human review (critical for claim amounts) |
| **Searchable PDF** | Archive compliance — make scanned documents searchable |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [02-layout-model.ipynb](02-layout-model.ipynb) | Extract tables and document structure from policy schedules |
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Build a confidence-based human review workflow |